In [3]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

Mounted at /gdrive


In [4]:
!ln -s "/gdrive/My Drive/Colab Notebooks/" "/content/source_code"

In [ ]:
!pip install neptune

In [1]:
from __future__ import print_function, division
from source_code.models.train import train_fn
from source_code.models.class_dataset import RoadSegDataset
from source_code.helper import *
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from torch.optim import lr_scheduler
from sklearn.model_selection import KFold
import torch.utils.data as data
import time
from source_code.models.unet.unet_class import Unet
import neptune

In [2]:
def test_model(loader, test_size, device, model_tt, loss_fn, threshold=0.5):
    running_loss = 0.0

    model_tt.eval()

    ground_truth = []
    predictions = []
    inference_time = []

    for inputs, masks in loader:
        inputs = inputs.to(device)
        inputs = inputs.float()
        masks = masks.to(device)
        masks = masks.squeeze()
        start = time.time()

        with torch.set_grad_enabled(False):

            outputs = model_tt(inputs).squeeze()
            loss = loss_fn(outputs, masks)
            preds = (outputs > threshold).float()

        ground_truth.append(masks.cpu().numpy())
        predictions.append(preds.cpu().numpy())
        inference_time.append(time.time() - start)

        running_loss += loss.item() * inputs.size(0)
        inference_time.append(time.time() - start)

    epoch_loss = running_loss / test_size
    print("Test loss={:.3f}".format(epoch_loss))

    return ground_truth, predictions, inference_time


In [3]:
lr_opt = 1e-4
batch_size = 10
num_epochs = 50
opti_opt = 'Adam'

"""
wd_opt = 0.00957460777341424
schedu_opt = 'Step'
step_size_opt = 10
gamma_opt = 0.2
"""

load_model = False
image_height = 400
image_width = 400

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [5]:
image_height = 400
image_width = 400

In [6]:
save_model = True
model_name = 'unet'
train_img_dir = "source_code/data/training/train_images"
train_masks_dir = "source_code/data/training/train_masks"
save_path = os.path.join("source_code",'models', model_name, 'Results')

In [7]:
loss_fn = nn.BCELoss()

In [8]:
import neptune

dataset = RoadSegDataset(train_img_dir, train_masks_dir)
print("Running experiments with: ", model_name)

report = {'iou': [], 'dice': [], 'f1': [], 'pixel accuracy': [], 'inference_time': []}

n_splits = 2
kf = KFold(n_splits=n_splits)

for fold, (train_ids, test_ids) in enumerate(kf.split(np.arange(dataset.__len__()))):
    # Split test_index into validation and test sets if necessary

    run = neptune.init_run(project="mehdihajoub/smartcervix",
                                           api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIyNTY0M2ZjNy05YjFmLTRlYTctYjg1ZS05ZmY2OTdjZjc1ZWYifQ==")
    run["sys/tags"].add(f"fold_{fold}")


    ground_truth = []
    predictions = []
    probabilities = []
    inf_time = []

    split = int(len(test_ids) * 0.5)
    val_ids, test_ids = test_ids[:split], test_ids[split:]

    dataset.__set_train_ids__(train_ids)
    dataset.__set_val_ids__(val_ids)
    dataset.__set_test_ids__(test_ids)

    print(len(train_ids), len(test_ids), len(val_ids))

    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)
    test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

    # Create data loaders
    train_loader = data.DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    val_loader = data.DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)
    test_loader = data.DataLoader(dataset, batch_size=batch_size, sampler=test_subsampler)

    save_flag = True

    # Initialize network
    model = Unet().to(device)

    # Model Optimizer

    optimizer_ft = optim.Adam(model.parameters(), lr=lr_opt)


    # Model Scheduler
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=num_epochs, eta_min=0)

    # Training the model and reporting the best epoch according to validation

    print(f'Training Started for fold number {fold} ')

    best_model, best_epoch = train_fn(train_loader, len(train_ids), val_loader, len(val_ids), device, model,
                                      loss_fn, optimizer_ft,
                                      scheduler, num_epochs, save_path, save_model, fold,run)

    run.stop()

    print(f'Training Ended for fold number {fold} ')

    print(f'Testing Started for fold number {fold}')

    _ground_truth, _predictions, _inf_time = test_model(test_loader, len(test_ids), device, best_model, loss_fn,
                                                        0.5)
    print(f'Testing Ended for fold number {fold}')

    ground_truth += _ground_truth
    predictions += _predictions
    inf_time += _inf_time

    # Assuming predictions and ground_truth are lists of lists
    flattened_predictions = [item for sublist in predictions for item in sublist]
    flattened_ground_truth = [item for sublist in ground_truth for item in sublist]

    # Now, flattened_predictions and flattened_ground_truth are single lists containing all the predictions and ground truths respectively

    iou, dice, pixel_acc, f = segmentation_metrics(flattened_ground_truth, flattened_predictions)

    report['iou'].append(iou)
    report['dice'].append(dice)
    report['pixel accuracy'].append(pixel_acc)
    report['f1'].append(f)
    report['inference_time'] += inf_time
    print("-------------------------------------")

if not os.path.exists(save_path):
    os.makedirs(save_path)
report_segmentation_performance(report, save_path)

print("Tbarkellah 3la si rachid")

Running experiments with:  unet


<ipython-input-8-37554d37430e>:14: NeptuneWarning: The following monitoring options are disabled by default in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', and 'capture_hardware_metrics'. To enable them, set each parameter to 'True' when initializing the run. The monitoring will continue until you call run.stop() or the kernel stops. Also note: Your source files can only be tracked if you pass the path(s) to the 'source_code' argument. For help, see the Neptune docs: https://docs.neptune.ai/logging/source_code/
  run = neptune.init_run(project="mehdihajoub/smartcervix",


https://app.neptune.ai/mehdihajoub/smartcervix/e/SMAR-44
50 25 25
Training Started for fold number 0 
Beginning of training
Epoch 0/49
train Loss: 0.6774
val Loss: 0.7125
Epoch 1/49
train Loss: 0.6287
val Loss: 0.7455
Epoch 2/49
train Loss: 0.5742
val Loss: 0.7772
Epoch 3/49
train Loss: 0.5050
val Loss: 0.7878
Epoch 4/49
train Loss: 0.4549
val Loss: 0.6687
Epoch 5/49
train Loss: 0.4050
val Loss: 0.5640
Epoch 6/49
train Loss: 0.3589
val Loss: 0.5355
Epoch 7/49
train Loss: 0.3380
val Loss: 0.4955
Epoch 8/49
train Loss: 0.3055
val Loss: 0.4884
Epoch 9/49
train Loss: 0.2849
val Loss: 0.4249
Epoch 10/49
train Loss: 0.2766
val Loss: 0.3987
Epoch 11/49
train Loss: 0.2685
val Loss: 0.4271
Epoch 12/49
train Loss: 0.2592
val Loss: 0.3281
Epoch 13/49
train Loss: 0.2471
val Loss: 0.3146
Epoch 14/49
train Loss: 0.2389
val Loss: 0.3186
Epoch 15/49
train Loss: 0.2320
val Loss: 0.2998
Epoch 16/49
train Loss: 0.2285
val Loss: 0.3045
Epoch 17/49
train Loss: 0.2266
val Loss: 0.2931
Epoch 18/49
train Loss

In [ ]:
!rm -r /content/source_code/Colab Notebooks

rm: cannot remove '/content/source_code/Colab': No such file or directory
rm: cannot remove 'Notebooks': No such file or directory


In [9]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import os

In [11]:
from source_code.models.unet.unet_class import Unet
device = "cpu"
model_pth = 'source_code/models/unet/Results/model_epoch39_fold_0.pth'

def load_model(model_path):

    model = Unet().to(device)
    model.load_state_dict(torch.load(model_path,map_location = torch.device(device)))
    model.eval()
    return model

def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.ToTensor(),
        #transforms.Normalize([0.33298134, 0.33009373, 0.29579783], [0.19390678, 0.18714967, 0.18701911])
    ])
    image = Image.open(image_path)
    image = transform(image)
    return image

def apply_threshold(prediction, threshold=0.5):
    prediction = prediction > threshold
    return prediction

def print_advancement(i):
    progress_e = 100 * i / 50
    print(f' Progress : {progress_e : .2f}%', end='\r')

# Load your trained model
model = load_model(model_pth)

# Directory containing your test images
test_images_root_dir = 'source_code/data/test_set_images'

# Directory to save the predictions
save_dir = 'source_code/models/unet/Results/preds'
os.makedirs(save_dir, exist_ok=True)

for i in range(1, 51):

    print_advancement(i)
    dir_path = os.path.join(test_images_root_dir, f'test_{i}')
    image_path = os.path.join(dir_path, f'test_{i}.png')  # Update if the naming is different

    image = preprocess_image(image_path)
    image = image.unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        prediction = model(image)

    # Convert prediction to an image format
    # Update this part based on your prediction format
    prediction_image = prediction.squeeze().cpu().numpy()
    binary_prediction = apply_threshold(prediction_image)

    # Convert binary prediction to image and save
    binary_prediction_image = Image.fromarray((binary_prediction * 255).astype(np.uint8))
    save_path = os.path.join(save_dir, f'prediction_{i}.png')
    binary_prediction_image.save(save_path)

In [5]:
from source_code.utils.mask_to_submission import *

In [6]:
submission_filename = 'dummy_submission.csv'
image_filenames = []
for i in range(1, 51):

    image_filename = f'/content/source_code/models/unet/Results/preds/prediction_{i}.png'
    image_filenames.append(image_filename)
masks_to_submission(submission_filename, *image_filenames)